# 07.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/07-forecasting/05-ml-forecasting/solutions.ipynb)

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

toy = pd.Series([30, 32, 28, 40, 25, 0, 18, 31, 33, 29, 42, 26],     # £k, starting on a Monday
                index=pd.date_range("2011-11-07", periods=12, freq="D"))

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

# The two sheets overlap: "Year 2009-2010" runs to 9 Dec 2010, "Year 2010-2011" starts 1 Dec 2010.
older = load_retail("Year 2009-2010")
newer = load_retail("Year 2010-2011")
raw = pd.concat([older[older["InvoiceDate"] < "2010-12-01"], newer], ignore_index=True)

def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
    return out.assign(Revenue=out["Quantity"] * out["Price"])

sales = clean_retail(raw)
sales = sales[sales["Quantity"] < 70_000]          # two one-off orders, both cancelled the same day
END = pd.Timestamp("2011-12-04")                    # last complete week (Mon 28 Nov - Sun 4 Dec)
sales = sales[sales["InvoiceDate"] < END + pd.Timedelta(days=1)]

rev = sales.set_index("InvoiceDate")["Revenue"]
daily = rev.resample("D").sum()                     # every calendar day, 0 when closed
weekly = rev.resample("W").sum()                    # weeks end on Sunday
monthly = rev.resample("ME").sum()                  # month-end dates
monthly = monthly[monthly.index <= "2011-11-30"]    # December 2011 has only 4 days: drop it
print(f"{len(sales):,} order lines, {daily.index[0].date()} to {daily.index[-1].date()}")
print(f"{len(daily)} days, {len(weekly)} weeks, {len(monthly)} months")

H = 5                                                   # forecast 5 weeks ahead
ORIGINS = pd.date_range("2011-04-03", periods=8, freq="4W-SUN")      # 3 Apr ... 16 Oct 2011

def backtest(forecast_fn, series=None, origins=ORIGINS, h=H):
    """Rolling-origin backtest: at each origin, fit on the past, forecast h weeks, score MAE."""
    series = weekly if series is None else series
    rows = []
    for origin in origins:
        train = series[series.index <= origin]
        test = series[series.index > origin].iloc[:h]
        pred = np.asarray(forecast_fn(train, len(test)))
        rows.append({"origin": origin.date(), "MAE": np.abs(test.values - pred).mean()})
    return pd.DataFrame(rows)

def naive(train, h):          return np.repeat(train.iloc[-1], h)
def mean_fc(train, h):        return np.repeat(train.mean(), h)
def seasonal_naive(train, h): return train.iloc[-52:-52 + h].values
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return train.iloc[-1] + slope * np.arange(1, h + 1)

from sklearn.ensemble import HistGradientBoostingRegressor

def daily_table(d):
    """Turn the daily series into a table. Every lag is >= 35 days, so 5 weeks ahead is honest."""
    t = pd.DataFrame({"y": d})
    t["lag_35"] = d.shift(35)
    t["mean_28_lag_35"] = d.shift(35).rolling(28).mean()
    t["dow"] = d.index.day_of_week
    t["month"] = d.index.month
    t["day_of_year"] = d.index.day_of_year
    t["days_to_xmas"] = (pd.to_datetime(d.index.year.astype(str) + "-12-25") - d.index).days
    return t

TABLE = daily_table(daily)
FEATURES = [c for c in TABLE.columns if c != "y"]

def gbm(train, h):
    """Train on daily rows up to the origin, predict the next h weeks of days, sum to weeks."""
    origin = train.index[-1]
    past = TABLE[TABLE.index <= origin].dropna()
    future = TABLE[TABLE.index > origin].iloc[: 7 * h]
    model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=10, random_state=0)
    model.fit(past[FEATURES], past["y"])
    pred = pd.Series(model.predict(future[FEATURES]), index=future.index).clip(lower=0)
    pred[future["dow"] == 5] = 0                       # ShopWave never trades on Saturdays
    return pred.resample("W").sum().values[:h]

768,341 order lines, 2009-12-01 to 2011-12-04
734 days, 105 weeks, 24 months


## Exercise 1

Saturday 12 Nov is day 6. lag_1 = Friday = 25; lag_7 = Saturday a week before: does not exist in the toy (the series starts on Monday 7th), so **NaN**; mean_3 of the three days before, Wed–Fri = (28 + 40 + 25) / 3 = **31**; dow = **5**.

In [2]:
t = pd.DataFrame({"lag_1": toy.shift(1), "lag_7": toy.shift(7),
                  "mean_3": toy.shift(1).rolling(3).mean().round(2), "dow": toy.index.day_of_week})
t.loc["2011-11-12"]

lag_1     25.0
lag_7      NaN
mean_3    31.0
dow        5.0
Name: 2011-11-12 00:00:00, dtype: float64

## Exercise 2

In [3]:
full = backtest(gbm)["MAE"].mean()
saved = FEATURES.copy()
FEATURES[:] = [f for f in saved if f not in ("days_to_xmas", "day_of_year")]
reduced = backtest(gbm)["MAE"].mean()
FEATURES[:] = saved
print(f"all features:            £{full:,.0f}")
print(f"without calendar-of-year £{reduced:,.0f}")

all features:            £34,678
without calendar-of-year £31,423


Removing `days_to_xmas` and `day_of_year` makes the model **better**, from about £34.7k to £31.4k, now ahead of seasonal naive (£32.4k). With only one previous year, features that identify the exact date let the trees memorise 2010's particular days, including one-off spikes and bank holidays that do not repeat on the same date. `month` and the 35-day lags carry the season more coarsely, which generalises better. More features is not automatically better, especially in time series with little history: always backtest a feature before keeping it.

## Exercise 3

In [4]:
wt = pd.DataFrame({"y": weekly, "lag_5": weekly.shift(5), "lag_52": weekly.shift(52),
                   "week_of_year": weekly.index.isocalendar().week.astype(int)})

def weekly_gbm(train, h):
    origin = train.index[-1]
    past = wt[wt.index <= origin].dropna(); future = wt[wt.index > origin].iloc[:h]
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=7,
                                      min_samples_leaf=5, random_state=0).fit(past.drop(columns="y"), past["y"])
    return m.predict(future.drop(columns="y"))

print(f"weekly GBM:     £{backtest(weekly_gbm)['MAE'].mean():,.0f}")
print(f"daily GBM:      £{backtest(gbm)['MAE'].mean():,.0f}")
print(f"seasonal naive: £{backtest(seasonal_naive)['MAE'].mean():,.0f}")
print("training rows at the first origin:", wt[wt.index <= ORIGINS[0]].dropna().shape[0])

weekly GBM:     £36,590


daily GBM:      £34,678
seasonal naive: £32,450
training rows at the first origin: 18


The weekly table has very few usable rows (a 52-week lag needs a year of history first), so the model is starved. It also hits the extrapolation trap from the lesson: its training rows contain no November. Daily rows give the model hundreds of examples instead of dozens.

## Exercise 4

In [5]:
rows = TABLE.dropna()
past = rows[rows.index <= "2011-10-30"]; future = TABLE[(TABLE.index > "2011-10-30")]
q90 = HistGradientBoostingRegressor(loss="quantile", quantile=0.9, max_iter=200, learning_rate=0.05,
                                    max_leaf_nodes=15, min_samples_leaf=10, random_state=0)
q90.fit(past[FEATURES], past["y"])
pred = pd.Series(q90.predict(future[FEATURES]), index=future.index).clip(lower=0)
pred[future["dow"] == 5] = 0
cmp = pd.DataFrame({"q90 forecast": pred.resample("W").sum(), "actual": weekly[weekly.index > "2011-10-30"]})
print(f"{(cmp['actual'] <= cmp['q90 forecast']).mean():.0%} of weeks at or below the 90% forecast")
(cmp / 1000).round(1)

100% of weeks at or below the 90% forecast


,q90 forecast,actual
InvoiceDate,,
2011-11-06,311.1,259.4
2011-11-13,317.4,263.7
2011-11-20,319.8,276.2
2011-11-27,316.7,248.2
2011-12-04,294.6,250.8


All five weeks came in below the 90% forecast, by roughly £45k–£70k each. That is more conservative than a true 90% line should be: summing daily 90th percentiles does not give a weekly 90th percentile, because not every day is high at once. Read it as a rough safety-stock ceiling rather than an exact 90% guarantee. Quantile forecasts are how many retailers set safety stock: plan to a high quantile for products where running out is costly, and to the median for everything else.